# Model Selection and Comparison — Making the Call Like a Professional

<hr>

<center>
<div>
<img src="https://raw.githubusercontent.com/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/main/notebooks/figures/mgmt_474_ai_logo_02-modified.png" width="200"/>
</div>
</center>

# <center><a class="tocSkip"></center>
# <center>MGMT47400 Predictive Analytics</center>
# <center>Professor: Davi Moreira </center>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davi-moreira/2026Summer_predictive_analytics_purdue_MGMT474/blob/main/notebooks/nb14_model_selection_protocol.ipynb)

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. Set up a **selection protocol** that evaluates 5 candidate models per spine on identical CV folds with a declared primary metric, before any results are seen.
2. Compute **Student's *t* 95% CI** on the champion's CV scores; apply the **CI-overlap rule** to decide whether the top model has earned displacement of the simpler runner-up.
3. Write a **champion selection memo** that justifies the choice in stakeholder language — what was compared, what was picked, and why.
4. Open the **locked test set exactly once per spine** and pronounce an INSIDE / ABOVE / BELOW verdict against the champion's CV CI.
5. Internalize the **singleness rule**: *two demo cases × one ceremony each = one ceremony per project*. Your own M2/M3/M4 work uses the test set ONCE — not twice.

---

> **📋 Participation Reminder:** This notebook contains **2 PAUSE-AND-DO exercises** — one champion selection memo per spine. Complete both before submitting your notebook.

---

## 💼 Why This Matters

Today is the payoff. Eleven notebooks of locking discipline (test set sealed, all evaluation via CV-with-CI on the training set, every new model benchmarked against the Week-2 reference) converge into a single structured ceremony: **declare the protocol, compare candidates, pick a champion, open the locked envelope.**

The State Health Department's review board is convening on the screening pipeline. HomeValue Analytics' deployment council is doing the same on the price-prediction model. Both want the same three things from your group:

1. A **defensible champion selection** — not "I tried a bunch of models and this one looked best", but "five candidates were declared in advance, evaluated under identical 5-fold CV folds with ROC-AUC (clf) / R² (reg) as the primary metric, and the champion was selected because its CV CI sat above the runner-up's by a clear margin (or, if CIs overlapped, because it was the simpler model)."
2. A **CV-CI 95% interval** on the champion's training-set performance — the headline number any stakeholder can quote.
3. A **single test-set evaluation** — the one and only authorized opening — that confirms (or contradicts) the CV-CI estimate.

The third bullet is the part this notebook makes visible. Before today every cell that touched the test set was off-limits. **In §6 the test set opens — once, for each of the two demo cases — and never again in the course.** nb15 onward goes back to CV-only evaluation on the training set. The test set is for the M3 milestone's final number, not for iteration.

A question that often comes up here is *"why is opening the test set such a big deal?"* Because every time you open the test set and let what you see influence what you do next, you have leaked the test set into model selection — which means the test set can no longer give you an unbiased estimate of generalization. The discipline is not statistical pedantry; it is the only mechanism by which the test-set point estimate has any meaning at all. nb14's ceremony exists to drive that point home in the most visible way possible: open the envelope, write the number down, close the envelope, never reopen.

### The singleness rule (read this carefully)

This notebook walks **two ceremonies** — one for classification, one for regression — because there are two demo cases. Your group's M2/M3/M4 final project is **one** of these cases (whichever your group picked at M0). For your project, you open the test set **ONCE**, not twice. Two ceremonies in this notebook is a pedagogical demo, not a precedent for your project work.

---

## 1. Setup — Imports, References, Helpers

The setup cell does the same five jobs as nb12 / nb13 plus one new utility: `compare_models_comprehensive()`, the comparison harness that takes a dict of candidate models and returns a sorted DataFrame of CV means with Student's *t* 95% CIs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, accuracy_score, f1_score,
                              r2_score, mean_squared_error, mean_absolute_error)
import time, warnings

warnings.filterwarnings('ignore')
pd.set_option('display.precision', 4)
plt.rcParams['figure.figsize'] = (10, 6)

RANDOM_SEED = 474
np.random.seed(RANDOM_SEED)
CLF_COLOR = '#1f77b4'
REG_COLOR = '#ff7f0e'
GREY      = '#999999'
GREEN     = '#2ca02c'
RED       = '#d62728'

# --- Week-2 references ---
reference_clf = Pipeline([('scaler', StandardScaler()),
                          ('clf',    LogisticRegression(C=1.0, random_state=RANDOM_SEED, max_iter=5000))])
reference_reg = Pipeline([('scaler', StandardScaler()),
                          ('reg',    LinearRegression())])

# --- Helper: CV-CI dot plot from previous notebooks ---
def plot_cv_ci(scores_dict, metric_name, title, ax, color=CLF_COLOR, k=5,
               highlight=None, highlight_color=GREEN):
    """Dot plot with 95% CIs; optionally highlight one model."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    rows = []
    for name, scores in scores_dict.items():
        m = float(np.mean(scores)); sd = float(np.std(scores, ddof=1))
        rows.append({'name': name, 'mean': m, 'half_w': t_crit * sd / np.sqrt(k)})
    df = pd.DataFrame(rows).sort_values('mean')
    colors = [highlight_color if (highlight is not None and r['name']==highlight) else color
              for _, r in df.iterrows()]
    for i, (_, r) in enumerate(df.iterrows()):
        ax.errorbar(r['mean'], i, xerr=r['half_w'], fmt='o', capsize=6, linewidth=2,
                    color=colors[i], markersize=10)
        ax.text(r['mean'] + r['half_w'] + (r['half_w']*0.2 if r['half_w']>0 else 0.001),
                i, f"{r['mean']:.4f}", va='center', fontsize=9)
    ax.set_yticks(range(len(df))); ax.set_yticklabels(df['name'])
    ax.set_xlabel(f'5-fold CV {metric_name} (mean ± 95% CI)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')

# --- Comparison harness ---
def compare_models_comprehensive(models_dict, X, y, cv, scoring, primary_metric):
    """Cross-validate each model on the same folds; return DataFrame with means, SDs, fit times."""
    rows = []
    for name, model in models_dict.items():
        t0 = time.time()
        cvr = cross_validate(model, X, y, cv=cv, scoring=scoring,
                             return_train_score=False, n_jobs=-1)
        fit_time = time.time() - t0
        row = {'Model': name, 'fit_time_s': fit_time}
        for m in scoring:
            row[f'{m}_mean'] = cvr[f'test_{m}'].mean()
            row[f'{m}_sd']   = cvr[f'test_{m}'].std(ddof=1)
            row[f'{m}_folds'] = cvr[f'test_{m}']
        rows.append(row)
    df = pd.DataFrame(rows)
    return df.sort_values(f'{primary_metric}_mean', ascending=False).reset_index(drop=True)

# --- Verdict helper for the ceremony ---
def verdict(test_score, cv_mean, cv_sd, k=5):
    """Compare a test-set point estimate to a 95% CV CI; return INSIDE/ABOVE/BELOW + message."""
    t_crit = stats.t.ppf(0.975, df=k - 1)
    half_w = t_crit * cv_sd / np.sqrt(k)
    lo, hi = cv_mean - half_w, cv_mean + half_w
    if test_score < lo:
        return 'BELOW', f'Test ({test_score:.4f}) is BELOW the CV CI ({lo:.4f}, {hi:.4f}) — model overfit the training data.'
    if test_score > hi:
        return 'ABOVE', f'Test ({test_score:.4f}) is ABOVE the CV CI ({lo:.4f}, {hi:.4f}) — pleasant surprise; investigate why CV underestimated.'
    return 'INSIDE', f'Test ({test_score:.4f}) is INSIDE the CV CI ({lo:.4f}, {hi:.4f}) — CV estimate held; ship the model.'

print("✓ Setup, references, helpers, harness, verdict function — all loaded")


---

## 2. The Model Selection Problem

Two failure modes of informal model selection ("just compare the numbers and pick the highest one"):

| Wrong | Right |
|---|---|
| Different CV splits per model | **Same CV folds** for every candidate |
| Different metric per model ("metric shopping") | **One declared primary metric**, supporting metrics for context |
| Pick by test-set score | Pick by CV; **test set is for verdict, not selection** |
| Try new candidates until something wins | **Lock the candidate roster before fitting any model** |
| Ignore fit time | Track fit time as a tie-breaker |
| No written rationale | Champion memo: *what, why, runner-up, when to revisit* |

The protocol below operationalizes the right column. Section 3 declares the candidate roster; Section 4 evaluates everything on identical CV folds; Section 5 visualizes; Section 6 writes the champion memo; Section 7 opens the locked test set.

---

## 3. Load Both Datasets — Two Locked Test Sets

Same 70/30 splits as nb11–nb13 with the same `random_state=RANDOM_SEED`. The CV scores you compute here will match (within fold-level fluctuation) the CV scores from previous notebooks because the splits are deterministic.

In [ ]:
data_clf = load_breast_cancer(as_frame=True)
X_clf, y_clf = data_clf.data, data_clf.target
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=RANDOM_SEED, stratify=y_clf
)
cv_clf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

data_reg = fetch_california_housing(as_frame=True)
X_reg, y_reg = data_reg.data, data_reg.target
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=RANDOM_SEED
)
cv_reg = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print(f'Classification: train {len(X_train_clf):>5} | test {len(X_test_clf):>5} (LOCKED until §7.1)')
print(f'Regression:     train {len(X_train_reg):>5} | test {len(X_test_reg):>5} (LOCKED until §7.2)')


---

## 4. Define the Candidate Roster — Five Per Spine

The candidate roster is **declared before any results are seen**. This rules out "model shopping" — the failure mode where you keep trying new models until one happens to beat the rest. Five candidates per spine; the same five families are represented across both:

| # | Family | Classification | Regression |
|---|---|---|---|
| 1 | **Week-2 reference (linear)** | LogReg(C=1.0) | OLS |
| 2 | **Sparse linear** | LogReg L1(C=0.1) | Lasso(α=0.01) |
| 3 | **Single tree** | DT(depth=3) | DT(depth=10) |
| 4 | **Bagged ensemble** | RF(n=100) | RF(n=100) |
| 5 | **Boosted ensemble** | GBM(default) | tuned GBM(lr=0.1, n=200, depth=5) |

Two notes on the roster:

- The **Week-2 reference** is candidate #1 because it is the floor every other candidate has to clear by a CI-clear margin to earn displacement.
- The **boosted ensemble** uses different hyperparameters per spine because nb13 showed that default-depth GBM does not lift past RF on regression — `max_depth=5` is what unlocks GBM's regression edge.

In [ ]:
clf_models = {
    'Week-2 ref: LogReg(C=1.0)': reference_clf,
    'LogReg L1 (C=0.1)':         Pipeline([('scaler', StandardScaler()),
                                            ('clf', LogisticRegression(penalty='l1', C=0.1, solver='liblinear',
                                                                       random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=3)':   DecisionTreeClassifier(max_depth=3, random_state=RANDOM_SEED),
    'Random Forest (100)':       RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    'Gradient Boosting (def)':   GradientBoostingClassifier(random_state=RANDOM_SEED),
}

reg_models = {
    'Week-2 ref: OLS':                    reference_reg,
    'Lasso (alpha=0.01)':                 Pipeline([('scaler', StandardScaler()),
                                                    ('reg', Lasso(alpha=0.01, random_state=RANDOM_SEED, max_iter=5000))]),
    'Decision Tree (depth=10)':           DecisionTreeRegressor(max_depth=10, random_state=RANDOM_SEED),
    'Random Forest (100)':                RandomForestRegressor(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1),
    'Gradient Boosting (lr=0.1, n=200, d=5)': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1,
                                                                         max_depth=5, random_state=RANDOM_SEED),
}

print(f"✓ {len(clf_models)} classification candidates, {len(reg_models)} regression candidates declared")
print("✓ Roster locked — no candidates added/removed after this cell")


---

## 5. Multi-Metric Reporting — CV Means with 95% CIs

For each spine, evaluate every candidate under the **same** 5-fold CV folds. Track the primary metric plus two supporting metrics:

- **Classification:** ROC-AUC (primary), accuracy, F1 (supporting)
- **Regression:** R² (primary), neg-RMSE, neg-MAE (supporting)

The CV-CI dot plot for the primary metric is the **central visual** of the selection ceremony. The companion table reports all metrics + fit times. The runner-up question — *"is the top model's CI clearly above the runner-up's?"* — is what the next section adjudicates.

In [ ]:
clf_scoring = ['roc_auc', 'accuracy', 'f1']
reg_scoring = ['r2', 'neg_root_mean_squared_error', 'neg_mean_absolute_error']

clf_results = compare_models_comprehensive(clf_models, X_train_clf, y_train_clf,
                                            cv=cv_clf, scoring=clf_scoring, primary_metric='roc_auc')
reg_results = compare_models_comprehensive(reg_models, X_train_reg, y_train_reg,
                                            cv=cv_reg, scoring=reg_scoring, primary_metric='r2')

print("=== CLASSIFICATION (ranked by CV ROC-AUC) ===")
print(clf_results[['Model', 'roc_auc_mean', 'roc_auc_sd', 'accuracy_mean', 'f1_mean', 'fit_time_s']]
      .to_string(index=False))
print()
print("=== REGRESSION (ranked by CV R²) ===")
print(reg_results[['Model', 'r2_mean', 'r2_sd',
                   'neg_root_mean_squared_error_mean', 'neg_mean_absolute_error_mean',
                   'fit_time_s']].to_string(index=False))

# Build dicts for plot_cv_ci
clf_dict = {row['Model']: row['roc_auc_folds'] for _, row in clf_results.iterrows()}
reg_dict = {row['Model']: row['r2_folds']      for _, row in reg_results.iterrows()}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_cv_ci(clf_dict, 'ROC-AUC', 'Classification — 5 candidates ranked', axes[0], color=CLF_COLOR)
plot_cv_ci(reg_dict, 'R²',      'Regression — 5 candidates ranked',     axes[1], color=REG_COLOR)
fig.suptitle('Five candidates per spine, ranked by CV mean of the primary metric',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


**Reading the output:**

The dot plots are the central artifact of the selection ceremony. Each candidate's mean is the dot; the horizontal bar is its 95% CI. The CI-overlap rule is now a visual question: *do the top model's bar and the runner-up's bar overlap?* If yes, statistical tie — pick the simpler model. If no, the top model has earned displacement.

On classification, the five CIs typically overlap heavily — Wisconsin breast cancer is small and mostly linear, so multiple candidates land within one SD of each other. The simpler models (Week-2 LogReg, LogReg L1) usually lead by a hair on mean, with the ensembles competitive but not strictly better.

On regression, the picture is dramatically more spread out. The Week-2 OLS reference sits at the bottom (~0.60). The single tree lifts above it. The random forest lifts above the tree. Tuned GBM (depth=5) sits at the top (~0.82) by a CI-clear margin. **The top candidate's CI does not overlap the runner-up's** — the ceremony has a clear winner.

**Key takeaway:** The dot plot turns the selection question into a visual one. The next section writes the champion memo for each spine.

---

## 6. Champion Selection Memo — Per Spine

The champion selection memo is the artifact a stakeholder reads. It has five parts, in order:

1. **Champion** — name of the chosen model, primary metric mean ± 95% CI.
2. **Runner-up** — name + score, plus the CI-overlap test result.
3. **Selection rationale** — *"chose champion because ..."* in stakeholder language. If CIs overlap with runner-up, the rationale is *"chose simpler model by parsimony"*.
4. **When to revisit** — concrete trigger conditions (e.g., "if CV ROC-AUC drops by more than 2 points on next quarter's data, retrain and re-evaluate").
5. **Risks** — known limitations (e.g., "interpretability lost vs the linear reference; LIME/SHAP needed for individual-prediction explanations").

The cell below writes the memo for each spine programmatically — same template, applied to whichever model topped the CV-CI dot plot.

In [ ]:
def write_champion_memo(results_df, primary_metric, problem_type, units_hint=''):
    """Generate a 5-part champion selection memo from the comparison DataFrame."""
    top, runner = results_df.iloc[0], results_df.iloc[1]
    t_crit = stats.t.ppf(0.975, df=4)
    top_half_w    = t_crit * top[f'{primary_metric}_sd']    / np.sqrt(5)
    runner_half_w = t_crit * runner[f'{primary_metric}_sd'] / np.sqrt(5)
    top_lo, top_hi       = top[f'{primary_metric}_mean'] - top_half_w,    top[f'{primary_metric}_mean'] + top_half_w
    runner_lo, runner_hi = runner[f'{primary_metric}_mean'] - runner_half_w, runner[f'{primary_metric}_mean'] + runner_half_w
    ci_overlap = not (top_lo > runner_hi or runner_lo > top_hi)

    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'CHAMPION SELECTION MEMO — {problem_type.upper()}')
    print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
    print(f'1. CHAMPION:   {top["Model"]}')
    print(f'              CV {primary_metric} = {top[f"{primary_metric}_mean"]:.4f}  '
          f'(95% CI: {top_lo:.4f}, {top_hi:.4f}) {units_hint}')
    print(f'2. RUNNER-UP: {runner["Model"]}')
    print(f'              CV {primary_metric} = {runner[f"{primary_metric}_mean"]:.4f}  '
          f'(95% CI: {runner_lo:.4f}, {runner_hi:.4f})')
    print(f'              CI overlap with champion: {ci_overlap}')
    if ci_overlap:
        print(f'3. RATIONALE: Top model and runner-up are statistically tied.')
        print(f'              Picking the simpler / faster of the two by parsimony.')
        # Tie-break by fit time
        if top['fit_time_s'] > runner['fit_time_s']:
            print(f'              -> SWITCHING champion to runner-up ({runner["Model"]}) because it fits faster.')
            top = runner
            top_lo, top_hi = runner_lo, runner_hi
    else:
        print(f'3. RATIONALE: Top model\'s CI ({top_lo:.4f}, {top_hi:.4f}) is clearly above')
        print(f'              runner-up\'s ({runner_lo:.4f}, {runner_hi:.4f}). CI-clear displacement.')
    print(f'4. WHEN TO REVISIT: If CV {primary_metric} drops below {top_lo:.4f} on retraining,')
    print(f'                    or if a new candidate family becomes available.')
    print(f'5. RISKS: Single-model deployment; consider ensemble averaging if score variance grows.')
    print()
    return top["Model"], top[f'{primary_metric}_mean'], top[f'{primary_metric}_sd']

# Memos for both spines
champ_clf_name, champ_clf_mean, champ_clf_sd = write_champion_memo(
    clf_results, 'roc_auc', 'classification (Wisconsin Breast Cancer)')
champ_reg_name, champ_reg_mean, champ_reg_sd = write_champion_memo(
    reg_results, 'r2', 'regression (California Housing)',
    units_hint='(R²; CV-RMSE in USD reported in §7.2)')


**Reading the output:**

Two memos, two champions. On classification the memo typically picks one of the linear candidates — LogReg(C=1.0) or LogReg L1 — because the top-by-mean's CI overlaps the runner-up's, and parsimony wins the tie. On regression the memo picks the tuned GBM by a CI-clear margin.

The memo template is the artifact you bring to the M3 milestone. Replace the dataset names, replace the candidate list, keep the five-part structure.

A question that often comes up here is *"if CIs overlap, do I always pick the simpler model?"* Almost always. The exception is when the simpler model has a known operational disadvantage (e.g., LogReg cannot handle missing values without imputation; the forest can). Default to the simpler model; deviate only with a stated reason.

---

## 📝 PAUSE-AND-DO Exercise 1 (clf, 5 minutes) — Write Your Champion Memo

**Task:** Look at the classification CV-CI dot plot from Section 5 and the auto-generated memo from Section 6. Write your **own** memo in the cell below, with two changes from the auto-generated version:

1. Justify the rationale in **stakeholder language** — translate "CI overlaps" into something the State Health Department's review board would actually say.
2. Add a **specific revisit trigger** tied to the operational context (e.g., "if false-negative rate at the deployed threshold exceeds 5% over a rolling 90-day window").

---

> 💡 **Gemini Prompt:** "Translate this technical champion-selection memo into stakeholder language for the State Health Department's review board. Champion: [your pick]. Runner-up: [from §5]. Justify in plain English; suggest one operational revisit trigger."


### YOUR CLASSIFICATION CHAMPION MEMO HERE:

**Champion:** *(your pick from §5)*

**Why this champion (in stakeholder language):**

*(your rationale)*

**Operational revisit trigger:**

*(your trigger)*

---

## 📝 PAUSE-AND-DO Exercise 2 (reg, 5 minutes) — Write Your Regression Champion Memo

**Task:** Same template, the regression spine. Look at the regression CV-CI dot plot and the auto-generated memo. Translate to **HomeValue Analytics' deployment council** language; add a revisit trigger tied to property-market dynamics.

---

> 💡 **Gemini Prompt:** "Translate this technical regression champion memo into stakeholder language for HomeValue Analytics' deployment council. Champion: [tuned GBM]. Runner-up: [random forest]. Convert R² to USD-RMSE when justifying. Suggest one operational revisit trigger tied to housing-market changes."


### YOUR REGRESSION CHAMPION MEMO HERE:

**Champion:** *(your pick from §5)*

**Why this champion (in stakeholder language):**

*(your rationale)*

**Operational revisit trigger:**

*(your trigger)*

---

## 7. Opening the Locked Test Set — The Ceremony

This is the moment the test set opens. Two ceremonies, two test sets, **one opening per case**. After this section, both test sets go back into the envelope and never reopen in this course.

The protocol is identical for both spines:

1. Refit the champion on the **full training set** (no fold leftover).
2. Predict on the locked test set.
3. Compute the test-set point estimate of the primary metric.
4. Compare against the champion's CV CI from §6.
5. Pronounce a verdict:
   - **INSIDE** the CI: CV estimate held; ship the model.
   - **ABOVE** the CI: pleasant surprise; investigate why CV underestimated (could be variance, could be lucky test).
   - **BELOW** the CI: the model overfit the training data; revisit candidates before shipping.

### 7.1 Classification Ceremony — Wisconsin Breast Cancer

The champion from §6 is `LogReg(C=1.0)` (or whichever the memo selected). Refit on `X_train_clf`, predict on `X_test_clf` — and that's the one and only authorized opening for the classification spine.

In [ ]:
# CEREMONY — CLASSIFICATION
# X_test_clf opens here, exactly once. After this cell, it goes back into the envelope.
champion_clf = clf_models[champ_clf_name]
champion_clf.fit(X_train_clf, y_train_clf)

# Get probability predictions for ROC-AUC and class predictions for accuracy/F1
y_proba_test_clf = champion_clf.predict_proba(X_test_clf)[:, 1]
y_pred_test_clf  = champion_clf.predict(X_test_clf)

test_auc      = roc_auc_score(y_test_clf, y_proba_test_clf)
test_accuracy = accuracy_score(y_test_clf, y_pred_test_clf)
test_f1       = f1_score(y_test_clf, y_pred_test_clf)

verdict_label_clf, verdict_msg_clf = verdict(test_auc, champ_clf_mean, champ_clf_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — CLASSIFICATION (Wisconsin Breast Cancer)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:           {champ_clf_name}')
print(f'CV ROC-AUC mean:    {champ_clf_mean:.4f}  (SD = {champ_clf_sd:.4f})')
print(f'Test ROC-AUC:       {test_auc:.4f}')
print(f'Test accuracy:      {test_accuracy:.4f}')
print(f'Test F1:            {test_f1:.4f}')
print(f'')
print(f'VERDICT: {verdict_label_clf}')
print(f'{verdict_msg_clf}')

# THE money plot for classification
t_crit = stats.t.ppf(0.975, df=4)
half_w = t_crit * champ_clf_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': CLF_COLOR, 'BELOW': RED}[verdict_label_clf]
ax.errorbar([champ_clf_mean], [0], xerr=[half_w], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3, label=f'CV mean ± 95% CI: {champ_clf_mean:.4f} ± {half_w:.4f}')
ax.scatter([test_auc], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_auc:.4f}  →  {verdict_label_clf}')
ax.axvline(champ_clf_mean - half_w, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_clf_mean + half_w, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('ROC-AUC')
ax.set_title(f'Classification ceremony — test ROC-AUC vs CV CI of the champion ({champ_clf_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

The triangle is the test-set point estimate; the dot is the CV mean; the dotted vertical lines mark the 95% CI bounds. If the triangle is between the dotted lines (INSIDE), the CV estimate held — ship the model. If the triangle is above the upper dotted line (ABOVE), the test set was kinder to the model than CV expected — investigate but do not panic. If the triangle is below the lower dotted line (BELOW), the model overfit the training data and the test-set number is the honest one.

Most of the time the verdict is INSIDE because CV-with-CI is well-calibrated under i.i.d. assumptions. ABOVE happens occasionally on small datasets (the test fold landed easy). BELOW is the dangerous case — it usually signals a leakage problem in the training pipeline that CV did not catch.

**The classification test set is now closed. It will not reopen in this course.**

---

### 7.2 Regression Ceremony — California Housing

Identical protocol. The champion is the tuned GBM. Refit on `X_train_reg`, predict on `X_test_reg`. One and only authorized opening for the regression spine.

In [ ]:
# CEREMONY — REGRESSION
# X_test_reg opens here, exactly once. After this cell, it goes back into the envelope.
champion_reg = reg_models[champ_reg_name]
champion_reg.fit(X_train_reg, y_train_reg)
y_pred_test_reg = champion_reg.predict(X_test_reg)

test_r2   = r2_score(y_test_reg, y_pred_test_reg)
test_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_test_reg))
test_mae  = mean_absolute_error(y_test_reg, y_pred_test_reg)

verdict_label_reg, verdict_msg_reg = verdict(test_r2, champ_reg_mean, champ_reg_sd)

print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'CEREMONY — REGRESSION (California Housing)')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'Champion:           {champ_reg_name}')
print(f'CV R² mean:         {champ_reg_mean:.4f}  (SD = {champ_reg_sd:.4f})')
print(f'Test R²:            {test_r2:.4f}')
print(f'Test RMSE:          {test_rmse:.4f}  (USD {test_rmse * 100_000:,.0f})')
print(f'Test MAE:           {test_mae:.4f}   (USD {test_mae  * 100_000:,.0f})')
print(f'')
print(f'VERDICT: {verdict_label_reg}')
print(f'{verdict_msg_reg}')

# Money plot for regression
t_crit = stats.t.ppf(0.975, df=4)
half_w_r = t_crit * champ_reg_sd / np.sqrt(5)
fig, ax = plt.subplots(figsize=(11, 4))
verdict_color = {'INSIDE': GREEN, 'ABOVE': REG_COLOR, 'BELOW': RED}[verdict_label_reg]
ax.errorbar([champ_reg_mean], [0], xerr=[half_w_r], fmt='o', capsize=10,
            color=GREY, markersize=12, linewidth=3,
            label=f'CV mean ± 95% CI: {champ_reg_mean:.4f} ± {half_w_r:.4f}')
ax.scatter([test_r2], [0], color=verdict_color, marker='^', s=300, zorder=5,
           label=f'Test point: {test_r2:.4f}  →  {verdict_label_reg}')
ax.axvline(champ_reg_mean - half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.axvline(champ_reg_mean + half_w_r, color=GREY, linestyle=':', alpha=0.5)
ax.set_yticks([]); ax.set_xlabel('R²')
ax.set_title(f'Regression ceremony — test R² vs CV CI of the champion ({champ_reg_name})',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper left'); ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()


**Reading the output:**

Same money plot, regression version. Triangle = test point; dot = CV mean; dotted lines = 95% CI bounds. The verdict carries the same three options. For California Housing with the tuned GBM, the verdict is usually INSIDE — the CV-with-CI estimate is well-calibrated and the held-out RMSE typically lands within USD 5K of the CV-RMSE.

Translating R² to USD-RMSE is the version stakeholders will quote. *"R² = 0.83"* is abstract; *"the model is typically off by USD 41,000 on a USD 200,000 home"* is a number HomeValue's deployment council can defend or contest in concrete terms.

**The regression test set is now closed. It will not reopen in this course.**

---

### 7.3 The Singleness Rule — Two Demos, One Ceremony Per Project

You just watched **two ceremonies** because there are two demo cases in this notebook. Your group's M2/M3/M4 final project is **one** of these cases (whichever your group picked at M0). For your project, you open the test set **ONCE**, not twice.

| What this notebook did | What your project does |
|---|---|
| Loaded breast cancer + California Housing | Loads your one project dataset |
| Ran 5-candidate selection on each spine | Runs 5-candidate selection on your one case |
| Wrote 2 champion memos | Writes 1 champion memo |
| Opened `X_test_clf` once + `X_test_reg` once | Opens your one project's `X_test` ONCE |
| Two INSIDE/ABOVE/BELOW verdicts (one per spine) | One INSIDE/ABOVE/BELOW verdict for the project |

The discipline is per-project, not per-notebook. Two ceremonies in this notebook is a pedagogical demo, not a precedent. **A single test set should never be opened twice in a project's lifetime.** If you find yourself wanting to "just check" the test set after seeing a CV result, that is the moment to close the laptop and re-read this section.

A question that often comes up here is *"what if my INSIDE verdict turns into a BELOW verdict on the test set — can I just add another candidate and re-run?"* No. Adding a candidate after seeing the test result is **selection on the test set**, which destroys the test set's role. Your options are:

1. **Accept the BELOW verdict** and report the test number honestly. The model is what the test set says it is; the CV was optimistic.
2. **Investigate the BELOW verdict** — usually it points to a leakage in your training pipeline. Fix the leakage, re-do CV (without touching the test set), and *only then* consider whether to re-open the test for a second ceremony in a different notebook (e.g., after the fix).
3. **Get more data** — extend the training set, re-do everything from nb09 forward.

What you cannot do is keep iterating on the same training/test split until the verdict turns favorable. That is the very pattern this whole course is designed to prevent.

---

## 8. Experiment Log Template

The last artifact: a reproducible experiment log. One row per ceremony, capturing everything a future analyst (or you, six months from now) needs to recreate today's selection without context.

In [ ]:
import datetime as dt
log_rows = [
    {
        'date':           dt.date.today().isoformat(),
        'spine':          'classification',
        'dataset':        'Wisconsin Breast Cancer (sklearn)',
        'n_train':        len(X_train_clf), 'n_test': len(X_test_clf),
        'cv_folds':       5, 'random_seed': RANDOM_SEED,
        'primary_metric': 'roc_auc',
        'champion':       champ_clf_name,
        'cv_mean':        round(champ_clf_mean, 4),
        'cv_sd':          round(champ_clf_sd, 4),
        'test_score':     round(test_auc, 4),
        'verdict':        verdict_label_clf,
    },
    {
        'date':           dt.date.today().isoformat(),
        'spine':          'regression',
        'dataset':        'California Housing (sklearn)',
        'n_train':        len(X_train_reg), 'n_test': len(X_test_reg),
        'cv_folds':       5, 'random_seed': RANDOM_SEED,
        'primary_metric': 'r2',
        'champion':       champ_reg_name,
        'cv_mean':        round(champ_reg_mean, 4),
        'cv_sd':          round(champ_reg_sd, 4),
        'test_score':     round(test_r2, 4),
        'verdict':        verdict_label_reg,
    },
]
log_df = pd.DataFrame(log_rows)
print('=== EXPERIMENT LOG ===')
print(log_df.to_string(index=False))

# Save to CSV — one row per ceremony, append-only is the recommended pattern
# log_df.to_csv('experiment_log.csv', mode='a', index=False, header=not Path('experiment_log.csv').exists())
print('\n💡 In M3, append your project\'s ceremony row to experiment_log.csv (one row per project, ever).')


---

## 9. Wrap-Up — Key Takeaways

**What landed today:**

1. **The selection protocol is a structured ceremony, not a spreadsheet exercise.** Five candidates per spine declared in advance, evaluated on identical CV folds, with one declared primary metric and supporting metrics for tie-breaking.
2. **The CV-CI dot plot is the central artifact.** CIs that overlap → simpler model wins. CIs that do not overlap → the top model has earned displacement.
3. **The champion memo is what stakeholders read.** Five parts: champion + CV CI, runner-up + overlap test, rationale in stakeholder language, when-to-revisit trigger, known risks.
4. **The locked test set opens exactly once per case.** Two ceremonies in this notebook is a pedagogical demo. Your project gets ONE ceremony.
5. **The verdict is INSIDE / ABOVE / BELOW** — not a number to debate, but a categorical pronouncement against the CV CI.

**Bridge to nb15 — Interpretation and Error Analysis:**

The champion is now committed. nb15 takes both committed champions forward (LogReg for classification, tuned GBM for regression) into the interpretation pass: permutation importance, partial dependence plots, segment-level error analysis, and the M3 milestone scaffold. The four-method importance heatmap from nb12 carries forward as the structural reference; PDPs add the *shape* dimension.

A question that often comes up at this point is *"if the CV-CI dot plot is so good at picking a champion, why does nb15 even exist?"* Because picking a champion is not the same as understanding what the champion learned. nb15's interpretation pass is the layer that translates the model into stakeholder-readable findings: *"this is what the model is paying attention to; this is where it fails; this is the segment-level fairness audit."* M3 is the deliverable; nb15 is the workshop.

---

## Participation Assignment Submission Instructions

### To Submit This Notebook:

1. **Complete both PAUSE-AND-DO exercises** — Exercise 1 (classification champion memo) and Exercise 2 (regression champion memo).
2. **Run All Cells** — `Runtime → Run all` to execute every cell including the two ceremonies.
3. **Save a Copy** — `File → Save a copy in Drive`, or download as `.ipynb`.
4. **Submit** — upload the `.ipynb` file to the Notebook 14 participation assignment on Brightspace.

### Before Submitting, Check:

- [ ] Both champion memos are written in stakeholder language
- [ ] Both money plots (test point vs CV CI) render with the verdict color coded
- [ ] You can defend the singleness rule in plain English

### Next Step:

- **Notebook 15** — Interpretation and Error Analysis (Day 15)

---

<center>

**Thank you!**

</center>